# 🚀 Chhattisgarh e-Procurement Chatbot - Fine-Tuning Notebook
This notebook fine-tunes **Llama 3 (8B)** or **Mistral 7B** on your custom bilingual 2,000-sample Chhattisgarh Store Purchase Rules dataset using **Unsloth**.

### ⏳ Estimated Training Time: ~15 to 20 minutes on a Free T4 GPU.

### 📋 Steps to Follow:
1. Set Colab Runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU).
2. Run all cells in sequence.
3. When prompted, upload the `mistral_train_dataset.jsonl` file created in your workspace.

### 1. Install Unsloth and Dependencies

In [ ]:
# Install Unsloth (specifically optimized for Colab GPUs)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

### 2. Mount Google Drive
We will save our final fine-tuned model (GGUF format) directly to Google Drive so it isn't lost when the Colab session terminates.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 3. Upload your Training Dataset
Run this cell and upload the `mistral_train_dataset.jsonl` file from your computer's `c:\cg-eproc-chatbot\data\` directory.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
dataset_filename = list(uploaded.keys())[0]
print(f"Uploaded dataset: {dataset_filename}")

### 4. Load Base Model (optimized 4-bit Llama-3-8B)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # None for auto-detection
load_in_4bit = True  # Use 4bit quantization to save memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

### 5. Setup PEFT (LoRA) Adapters
We configure Parameter-Efficient Fine-Tuning (PEFT) targeting the self-attention projection weights of the model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

### 6. Process the Dataset
Apply the model's chat template structure to the training entries.

In [ ]:
import json
from datasets import Dataset

dataset_data = []
with open(dataset_filename, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            dataset_data.append(json.loads(line))

def format_prompts(examples):
    texts = []
    for messages in examples["messages"]:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return { "text" : texts }

raw_dataset = Dataset.from_list(dataset_data)
dataset = raw_dataset.map(format_prompts, batched=True)
print(f"Loaded and formatted {len(dataset)} training samples.")

### 7. Configure SFTTrainer and Run Fine-Tuning

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 300, # ~3 epochs on batch size 8 for 2000 samples
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("Starting fine-tuning...")
trainer_stats = trainer.train()
print("Fine-tuning complete!")

### 8. Export the Merged Model to GGUF (for offline/Ollama use)
We merge our trained adapter weights with the base model and convert it to a 4-bit quantized GGUF file. This file will be saved directly into your Google Drive under `MyDrive/cg_procurement_model.gguf`.

In [ ]:
save_path = "/content/drive/MyDrive/cg_procurement_model"
print(f"Saving merged GGUF model directly to Google Drive: {save_path}.gguf")

# unsloth automatically handles GGUF quantization
model.save_pretrained_gguf(
    save_path,
    tokenizer,
    quantization_method = "q4_k_m" # Quantized standard format
)
print("🎉 Model GGUF saved successfully to your Google Drive!")